# 02 — Data Cleaning and Feature Engineering
### Household Electricity Consumption Prediction Using ANN
**Author:** Roshan Langhi | B.Tech Computer Engineering

---
### 📌 Objective
Clean the dataset, handle missing records, combine Date/Time features, and extract temporal attributes (Hour, Day, Month, DayOfWeek).



In [ ]:
import os
import pandas as pd
import numpy as np

RAW_DATA_PATH = os.path.join("..", "data", "raw", "household_power_consumption.txt")
PROCESSED_DATA_PATH = os.path.join("..", "data", "processed", "cleaned_data.csv")

os.makedirs(os.path.dirname(PROCESSED_DATA_PATH), exist_ok=True)



### 1. Load Data


In [ ]:
df = pd.read_csv(
    RAW_DATA_PATH,
    sep=';',
    na_values='?',
    low_memory=False
)
print(f"Loaded {len(df):,} records.")



### 2. Combine Date and Time into Datetime


In [ ]:
df['Datetime'] = pd.to_datetime(
    df['Date'] + ' ' + df['Time'],
    format='%d/%m/%Y %H:%M:%S',
    errors='coerce'
)
df.dropna(subset=['Datetime'], inplace=True)
df.sort_values('Datetime', inplace=True)
df.drop(columns=['Date', 'Time'], inplace=True)
df.head()



### 3. Handle Missing Values and Convert Types


In [ ]:
numeric_cols = [
    'Global_active_power', 'Global_reactive_power', 'Voltage',
    'Global_intensity', 'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3'
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Drop remaining missing values
before_drop = len(df)
df.dropna(inplace=True)
print(f"Removed {before_drop - len(df):,} records with missing numeric values.")
print(f"Remaining clean records: {len(df):,}")



### 4. Temporal Feature Engineering
Extract hour, day, month, and day of week from Datetime to provide the ANN with periodic context.



In [ ]:
df['Hour'] = df['Datetime'].dt.hour
df['Day'] = df['Datetime'].dt.day
df['Month'] = df['Datetime'].dt.month
df['DayOfWeek'] = df['Datetime'].dt.dayofweek

df[['Datetime', 'Hour', 'Day', 'Month', 'DayOfWeek', 'Global_active_power']].head()



### 5. Check Duplicates and Export Cleaned Dataset Sample


In [ ]:
print(f"Duplicate Datetime rows: {df['Datetime'].duplicated().sum()}")

# To maintain lightweight processing and reproducible development, export a representative 100,000 row sample
sample_df = df.sample(n=min(100000, len(df)), random_state=42).sort_values('Datetime')
sample_df.to_csv(PROCESSED_DATA_PATH, index=False)

print(f"Exported cleaned data sample to: {PROCESSED_DATA_PATH}")
print(f"Shape: {sample_df.shape}")

